## **Piper TTS Model**

Piper will be used as a pre-trained model without training or fine_tuning since it only converts the text it recievs from the LLM into speech. This notebook is a **testing pass**, not a training pipeline for checking output quality, correctness on edge cases and latency, then wrapping it for integration. There is no train/val/test split because there is no weigths being updated.

In [1]:
!pip install -q piper-tts

In [2]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id = "rhasspy/piper-voices",
    filename = "en/en_US/lessac/medium/en_US-lessac-medium.onnx"
)
#filename = "en/en_GB/alan/medium/en_GB-alan-medium.onnx"
config_path = hf_hub_download(
    repo_id = "rhasspy/piper-voices",
    filename = "en/en_US/lessac/medium/en_US-lessac-medium.onnx.json"
)

## **Loading the Model**

In [3]:
from piper import PiperVoice
voice = PiperVoice.load(model_path, config_path)
print("Sample rate:", voice.config.sample_rate)

Sample rate: 22050


Checking wether the model runs at all.

In [4]:
import wave
from IPython.display import Audio, display

with wave.open("sanity_check.wav", "wb") as wav_file:
  voice.synthesize_wav("Hi there! Let's practice talking about your last vacation.", wav_file)

  display(Audio("sanity_check.wav"))
  print("Sanity check complete.")

Sanity check complete.


A small set that looks like real AI replies from the app's topics.

In [5]:
test_sentences = {
    "topic_travel":        "That sounds like a great trip — where are you flying from?",
    "topic_job_interview": "Tell me about a time you solved a difficult problem at work.",
    "topic_ordering_food": "Would you like that with fries or a side salad instead?",
    "numbers":             "The meeting is at 3:45 and there are 12 people coming.",
    "contraction":         "I don't think that's going to work, but let's try it anyway.",
    "name":                "Nice to meet you, my name is Sarah, what's yours?",
    "question":            "Have you ever visited another country before?",
    "long_sentence":       "So, if I understand you correctly, you're saying that you'd "
                            "rather practice speaking about your daily routine than about "
                            "your favorite hobbies, is that right?"
}

## **Generating Audio & Measuring Latency**

In [6]:
import time

results = []
for label, sentence in test_sentences.items():
  filename = f"tts_test_{label}.wav"

  start = time.time()
  with wave.open(filename, "wb") as wav_file:
    voice.synthesize_wav(sentence, wav_file)
  elapsed = time.time() - start

  results.append({"label": label, "text": sentence, "seconds": round(elapsed, 2)})
  print(f"{label:20s} | {elapsed:5.2f}s | {sentence}")

  display(Audio(filename))

topic_travel         |  1.10s | That sounds like a great trip — where are you flying from?


topic_job_interview  |  0.77s | Tell me about a time you solved a difficult problem at work.


topic_ordering_food  |  0.48s | Would you like that with fries or a side salad instead?


numbers              |  0.58s | The meeting is at 3:45 and there are 12 people coming.


contraction          |  0.53s | I don't think that's going to work, but let's try it anyway.


name                 |  0.48s | Nice to meet you, my name is Sarah, what's yours?


question             |  0.37s | Have you ever visited another country before?


long_sentence        |  1.64s | So, if I understand you correctly, you're saying that you'd rather practice speaking about your daily routine than about your favorite hobbies, is that right?


## **Comparing Model Voices**

In [7]:
voices_to_try =[
     ("en/en_US/lessac/medium/en_US-lessac-medium.onnx", "lessac"),
    ("en/en_US/amy/medium/en_US-amy-medium.onnx", "amy"),
    ("en/en_GB/alan/medium/en_GB-alan-medium.onnx", "alan")
]

sample_text = "Let's talk about your favourite hobby."

for onnx_path, name in voices_to_try:
  m_path = hf_hub_download(repo_id="rhasspy/piper-voices", filename=onnx_path)
  c_path = hf_hub_download(repo_id="rhasspy/piper-voices", filename=onnx_path+".json")
  v = PiperVoice.load(m_path, c_path)

  filename = f"voice_test_{name}.wav"
  with wave.open(filename, "wb") as wav_file:
    v.synthesize_wav(sample_text, wav_file)

  print(f"Voice: {name}")
  display(Audio(filename))

Voice: lessac


Voice: amy


Voice: alan


Chosen voice: Lessac - clearest pronunciation and natural pacing.

In [8]:
CHOSEN_MODEL_PATH =model_path
CHOSEN_CONFIG_PATH =config_path

_voice = PiperVoice.load(CHOSEN_MODEL_PATH, CHOSEN_CONFIG_PATH)

def synthesize(text:str) -> bytes:
  import io
  buffer = io.BytesIO()
  with wave.open(buffer, "wb") as wav_file:
    _voice.synthesize_wav(text, wav_file)
  return buffer.getvalue()

_ = synthesize("This is a test of the wrapped function.")
print("synthesize() is ready to use.")

synthesize() is ready to use.
